# UnderCover: comparing 100 OpenRouter models and two agent designs

This notebook converts the reusable model sweep into an experiment that is easier to learn from and run interactively.

It compares two protocols on UnderCover claims:

1. **Direct**: the model receives the policy, claim, and policy wording, then returns structured JSON.
2. **Agent**: the model receives a claim, calls tools to retrieve the policy and record its decision, and then finishes.

The handler answer key is used **only after** a model responds. It is never included in a prompt. The default settings are deliberately small so you can inspect the setup without unexpectedly spending money or exhausting OpenRouter rate limits.

In [ ]:
# Imports used by the experiment. The async libraries let us run several
# independent model requests at once while respecting a concurrency limit.
import asyncio
import json
import os
import re
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import httpx
from dotenv import load_dotenv
from openai import AsyncOpenAI

# This notebook is normally started from the repository root in VS Code.
# The fallback also works if the notebook is launched from its own directory.
ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = Path("../..").resolve()
DATA = ROOT / "data"
OPENROUTER_MODELS_URL = "https://openrouter.ai/api/v1/models"
OPENROUTER_CHAT_URL = "https://openrouter.ai/api/v1"
MAX_RETRIES = 2

load_dotenv(ROOT / ".env", override=True)
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
if not OPENROUTER_API_KEY:
    raise RuntimeError("OPENROUTER_API_KEY is missing from .env")
print("OpenRouter API key found")

## 1. Load UnderCover data

The claims and policies are the input to the experiment. The held-out decisions are loaded separately for scoring and are not passed to either model.

In [ ]:
def load_json(name: str) -> Any:
    """Load one repository data file."""
    return json.loads((DATA / name).read_text(encoding="utf-8"))


claims = load_json("claims.json")
policies = {row["policy_id"]: row for row in load_json("policies.json")}
labels = {row["claim_id"]: row for row in load_json("evals/decisions.json")}
policy_wording = (DATA / "policy_wording.md").read_text(encoding="utf-8")

print(f"{len(claims)} claims, {len(policies)} policies, {len(labels)} answer-key labels")

## 2. Discover and select models

OpenRouter's catalog changes over time. We fetch it at runtime, ignore invalid or negative prices, put free models first, and then add models below the configured price caps.

Prices from OpenRouter are dollars per token. The notebook displays them as dollars per million tokens.

In [ ]:
@dataclass(frozen=True)
class ModelInfo:
    """The model identifier and its OpenRouter input/output prices."""

    model_id: str
    prompt_price: float
    completion_price: float

    @property
    def is_free(self) -> bool:
        return self.prompt_price == 0 and self.completion_price == 0

    @property
    def blended_price(self) -> float:
        return self.prompt_price + self.completion_price


def money(value: Any) -> float:
    """Convert a catalog price to dollars per token; reject invalid prices."""
    try:
        parsed = float(value or 0)
    except (TypeError, ValueError):
        return float("inf")
    return parsed if parsed >= 0 else float("inf")


async def fetch_models(api_key: str) -> list[ModelInfo]:
    """Fetch currently available models with usable pricing."""
    headers = {"Authorization": f"Bearer {api_key}"}
    async with httpx.AsyncClient(timeout=30) as client:
        response = await client.get(OPENROUTER_MODELS_URL, headers=headers)
        response.raise_for_status()

    models = []
    for raw in response.json().get("data", []):
        pricing = raw.get("pricing") or {}
        prompt_price = money(pricing.get("prompt"))
        completion_price = money(pricing.get("completion"))
        if raw.get("id") and prompt_price != float("inf") and completion_price != float("inf"):
            models.append(ModelInfo(raw["id"], prompt_price, completion_price))

    return sorted(models, key=lambda model: (not model.is_free, model.blended_price, model.model_id))


def select_models(
    models: list[ModelInfo],
    count: int,
    max_prompt_price: float,
    max_completion_price: float,
) -> list[ModelInfo]:
    """Choose free models first, followed by models under both price caps."""
    selected = [
        model for model in models
        if model.is_free
        or (
            model.prompt_price <= max_prompt_price / 1_000_000
            and model.completion_price <= max_completion_price / 1_000_000
        )
    ]
    if len(selected) < count:
        raise RuntimeError(
            f"Only {len(selected)} models meet the price caps; {count} are required."
        )
    return selected[:count]

In [ ]:
# This cell makes one catalog request, but it does not call any chat model.
all_models = await fetch_models(OPENROUTER_API_KEY)
selected_models = select_models(
    all_models,
    count=100,
    max_prompt_price=1.0,       # dollars per million input tokens
    max_completion_price=5.0,   # dollars per million output tokens
)

print(f"Selected {len(selected_models)} models ({sum(m.is_free for m in selected_models)} free).")
for model in selected_models[:10]:
    print(
        f"{model.model_id}: "
        f"${model.prompt_price * 1_000_000:.3f}/M in, "
        f"${model.completion_price * 1_000_000:.3f}/M out"
    )
print("..." if len(selected_models) > 10 else "")

## 3. Build prompts and score responses

The direct protocol asks for a small, stable JSON shape. The scoring function checks the decision and allows a payout difference of up to one dollar, matching the course experiment.

In [ ]:
def make_prompt(claim: dict[str, Any], policy: dict[str, Any], wording: str) -> str:
    """Build the complete context sent to a direct model."""
    return (
        "# Policy record\n\n"
        f"{json.dumps(policy, indent=2)}\n\n"
        "# Claim\n\n"
        f"{json.dumps(claim, indent=2)}\n\n"
        "# Policy wording\n\n"
        f"{wording}\n\n"
        "Return JSON with exactly these keys: decision (approve, decline, or refer), "
        "payout (number), reason (string), and reference (string)."
    )


def extract_json(text: str) -> dict[str, Any]:
    """Parse a JSON object, tolerating explanatory text around it."""
    try:
        value = json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if not match:
            raise ValueError("Model did not return a JSON object") from None
        value = json.loads(match.group(0))
    if not isinstance(value, dict):
        raise ValueError("Model JSON response was not an object")
    return value


def claim_is_right(response: dict[str, Any], label: dict[str, Any]) -> bool:
    """Compare a model response with the handler label after the call."""
    decision = str(response.get("decision", "")).lower()
    try:
        payout = float(response.get("payout", 0))
    except (TypeError, ValueError):
        return False
    return decision == label["decision"] and abs(payout - float(label["payout"])) <= 1

## 4. Define the agent tools

The agent protocol gives the model tools instead of placing the policy record directly in its user message. The local tool responses below simulate backend operations; they do not expose the answer key.

In [ ]:
def agent_tools() -> list[dict[str, Any]]:
    """Describe the tools available to the tool-using model."""
    return [
        {
            "type": "function",
            "function": {
                "name": "get_policy",
                "description": "Retrieve the customer's policy record before deciding.",
                "parameters": {"type": "object", "properties": {}, "additionalProperties": False},
            },
        },
        {
            "type": "function",
            "function": {
                "name": "record_decision",
                "description": "Record the final claim decision and payout.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "decision": {"type": "string", "enum": ["approve", "decline", "refer"]},
                        "payout": {"type": "number"},
                        "reason": {"type": "string"},
                        "reference": {"type": "string"},
                    },
                    "required": ["decision", "payout", "reason", "reference"],
                    "additionalProperties": False,
                },
            },
        },
    ]

## 5. Call one model

`call_model` supports both protocols. It retries transient failures twice, records latency and token usage, and treats a tool call to `record_decision` as the agent's final answer.

In [ ]:
async def call_model(  # noqa: PLR0913, PLR0917
    client: AsyncOpenAI,
    model: ModelInfo,
    claim: dict[str, Any],
    policy: dict[str, Any],
    wording: str,
    mode: str,
) -> tuple[dict[str, Any], int, float]:
    """Run one direct request or one tool-using agent conversation."""
    system = (
        "You are an UnderCover claims handler. Follow the policy wording exactly. "
        "Do not invent facts. Use the available tools and make one final decision.\n\n"
        f"Policy wording:\n{wording}"
    )
    if mode == "direct":
        messages: list[dict[str, Any]] = [
            {"role": "system", "content": system},
            {"role": "user", "content": make_prompt(claim, policy, wording)},
        ]
    else:
        messages = [
            {"role": "system", "content": system},
            {
                "role": "user",
                "content": (
                    f"Claim:\n{json.dumps(claim, indent=2)}\n\n"
                    "Call get_policy, apply the policy wording, then "
                    "call record_decision."
                ),
            },
        ]

    started = time.perf_counter()
    for attempt in range(MAX_RETRIES + 1):
        try:
            response = await client.chat.completions.create(
                model=model.model_id,
                messages=messages,
                tools=None if mode == "direct" else agent_tools(),
                tool_choice="auto" if mode == "agent" else None,
                temperature=0,
            )
            usage = int(response.usage.total_tokens) if response.usage else 0
            message = response.choices[0].message
            if mode == "direct":
                return (
                    extract_json(message.content or ""),
                    usage,
                    time.perf_counter() - started,
                )
            for tool_call in message.tool_calls or []:
                if tool_call.function.name == "get_policy":
                    messages.extend(
                        [
                            message.model_dump(exclude_none=True),
                            {
                                "role": "tool",
                                "tool_call_id": tool_call.id,
                                "content": json.dumps(policy),
                            },
                        ]
                    )
                elif tool_call.function.name == "record_decision":
                    return (
                        extract_json(tool_call.function.arguments),
                        usage,
                        time.perf_counter() - started,
                    )
            messages.append(message.model_dump(exclude_none=True))
            messages.append(
                {
                    "role": "user",
                    "content": "Use record_decision now with your final answer.",
                }
            )
        except Exception:
            if attempt == MAX_RETRIES:
                raise
            await asyncio.sleep(2**attempt)
    raise RuntimeError("Model call ended without a result")

## 6. Run the sweep

The semaphore limits concurrent requests. With 100 models, one claim, and both protocols this creates 200 requests. With five claims it creates 1,000 requests, so check your OpenRouter limits first.

In [ ]:
async def run_sweep(  # noqa: PLR0913, PLR0917
    models: list[ModelInfo],
    claims: list[dict[str, Any]],
    labels: dict[str, dict[str, Any]],
    policies: dict[str, dict[str, Any]],
    wording: str,
    api_key: str,
    mode: str,
    concurrency: int,
) -> list[dict[str, Any]]:
    """Run every selected model over every selected claim."""
    client = AsyncOpenAI(api_key=api_key, base_url=OPENROUTER_CHAT_URL)
    semaphore = asyncio.Semaphore(concurrency)

    async def run_one(
        model: ModelInfo, claim: dict[str, Any], protocol: str
    ) -> dict[str, Any]:
        async with semaphore:
            try:
                response, tokens, seconds = await call_model(
                    client,
                    model,
                    claim,
                    policies[claim["policy_id"]],
                    wording,
                    protocol,
                )
                return {
                    "model": model.model_id,
                    "protocol": protocol,
                    "claim_id": claim["claim_id"],
                    "right": claim_is_right(response, labels[claim["claim_id"]]),
                    "payout": response.get("payout"),
                    "decision": response.get("decision"),
                    "tokens": tokens,
                    "seconds": round(seconds, 3),
                    "error": "",
                }
            except Exception as error:
                return {
                    "model": model.model_id,
                    "protocol": protocol,
                    "claim_id": claim["claim_id"],
                    "right": False,
                    "payout": None,
                    "decision": None,
                    "tokens": 0,
                    "seconds": 0,
                    "error": f"{type(error).__name__}: {error}",
                }

    protocols = ["direct", "agent"] if mode == "both" else [mode]
    jobs = [
        run_one(model, claim, protocol)
        for model in models
        for protocol in protocols
        for claim in claims
    ]
    return await asyncio.gather(*jobs)

## 7. Run a small experiment first

This is the safe starting point: five selected models, one claim, and both protocols. Change `MODEL_COUNT`, `CLAIM_COUNT`, and `MODE` only after confirming your account limits and budget.

Set `RUN_EXPERIMENT = True` to make chat requests. It is `False` by default.

In [ ]:
MODEL_COUNT = 5
CLAIM_COUNT = 1
MODE = "both"  # "direct", "agent", or "both"
CONCURRENCY = 4
RUN_EXPERIMENT = False
OUTPUT = ROOT / "community_contributions/aparnamantha/model_sweep_results.json"

models_for_run = selected_models[:MODEL_COUNT]
claims_for_run = claims if CLAIM_COUNT == 0 else claims[:CLAIM_COUNT]

if not RUN_EXPERIMENT:
    print(
        f"Preview: {len(models_for_run)} models x {len(claims_for_run)} claims "
        f"x {2 if MODE == 'both' else 1} protocol(s)."
    )
    print("Set RUN_EXPERIMENT = True to call OpenRouter.")
else:
    results = await run_sweep(
        models_for_run,
        claims_for_run,
        labels,
        policies,
        policy_wording,
        OPENROUTER_API_KEY,
        MODE,
        max(1, CONCURRENCY),
    )
    OUTPUT.write_text(json.dumps(results, indent=2), encoding="utf-8")
    correct = sum(row["right"] for row in results)
    print(f"Saved {len(results)} rows to {OUTPUT}")
    print(f"Accuracy: {correct}/{len(results)}")

## 8. Summarize results

Run this cell after an experiment. It groups by protocol and model so you can compare accuracy, average latency, and average token usage.

In [ ]:
if RUN_EXPERIMENT:
    from collections import defaultdict

    grouped = defaultdict(list)
    for row in results:
        grouped[(row["protocol"], row["model"])].append(row)

    summary = []
    for (protocol, model), rows in grouped.items():
        summary.append(
            {
                "protocol": protocol,
                "model": model,
                "correct": sum(row["right"] for row in rows),
                "total": len(rows),
                "accuracy": round(sum(row["right"] for row in rows) / len(rows), 3),
                "avg_seconds": round(sum(row["seconds"] for row in rows) / len(rows), 3),
                "avg_tokens": round(sum(row["tokens"] for row in rows) / len(rows)),
                "errors": sum(bool(row["error"]) for row in rows),
            }
        )

    for row in sorted(summary, key=lambda item: (-item["accuracy"], item["avg_seconds"])):
        print(row)
else:
    print("No results yet. Set RUN_EXPERIMENT = True and rerun the previous cell.")

## Notes and limitations

- OpenRouter's free catalog and routing behavior change, so results are snapshots rather than permanent rankings.
- A free model may still have daily or per-minute request limits.
- The experiment compares decision and payout, not the quality of the explanation.
- The answer key remains outside all prompts, but the notebook itself loads it for evaluation.
- Never open or include held-out evaluation data in a model prompt.